In [1]:
# import libraries
from openai import OpenAI
import time
from dotenv import load_dotenv
import os
import sys
from pathlib import Path
import torch
import json
import random
import re
import sys
from sklearn.metrics import classification_report as sklearn_classification_report
from seqeval.metrics import classification_report as seqeval_classification_report

# set path to project root and import custom classes and functions
base_path = Path.cwd() / "../../../"
sys.path.append(str(base_path.resolve()))
from utils.evaluation import extract_spans, mention_level_evaluation

In [2]:
def create_llm_annotations(text, spans):

    # sort the spans according to their start and end index
    spans = sorted(spans, key=lambda x: x["start"])

    # store the text and set index variable
    llm_text = ""
    last_idx = 0

    # loop through spans and add the span with custom characters
    for span in spans:
        llm_text += text[last_idx:span["start"]]
        llm_text += f"@@{text[span["start"]:span["end"]]}##"
        last_idx = span["end"]

    # add the rest of the text
    llm_text += text[last_idx:]

    return llm_text

def llm_output_to_bio(annotated_text):

    # split words via a regex
    words = re.findall(r"@@.*?##|\w+|'\w+|[^\w\s]", annotated_text)

    # empty list to store the bio tags
    bio_tags = []

    # loop through all words
    for word in words:

        # if it is an annotated span, split words and assign bio labels
        if word.startswith("@@") and word.endswith("##"):
            entity_text = word[2:-2]
            entity_words = re.findall(r"\w+|'\w+|[^\w\s]", entity_text)
            for i, t in enumerate(entity_words):
                tag = "B-sg" if i == 0 else "I-sg"
                bio_tags.append((t, tag))
        else:
            # otherwise assign O tag
            bio_tags.append((word, "O"))

    return bio_tags

In [3]:
# initialize empty dataset list
training_data = []
validation_data = []

with open("../../../01_data/training_validation_set/training_set.json", "r") as f:
    raw_training_data = json.load(f)

with open("../../../01_data/training_validation_set/validation_set.json", "r") as f:
    raw_val_data = json.load(f)

# loop through all sentences in the data
for task in raw_training_data:
    # get the sentence and all annotations
    text = task["sentence"]
    spans = task["annotations"]
    labels = [annotation["text"] for annotation in spans]
   
    llm_text = create_llm_annotations(text, spans)
    bio_tags = llm_output_to_bio(llm_text)

    # add everything to the dataset list
    training_data.append({
        "text": text,
        "labels": labels,
        "llm_text": llm_text,
        "bio_tags": bio_tags
    })

# loop through all sentences in the data
for task in raw_val_data:
    # get the sentence and all annotations
    text = task["sentence"]
    spans = task["annotations"]
    labels = [annotation["text"] for annotation in spans]
   
    llm_text = create_llm_annotations(text, spans)
    bio_tags = llm_output_to_bio(llm_text)

    # add everything to the dataset list
    validation_data.append({
        "text": text,
        "labels": labels,
        "llm_text": llm_text,
        "bio_tags": bio_tags
    })

In [28]:
# load the model
load_dotenv()  # reads .env file
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [23]:
# retrieve some few-shot examples
# create some few-shot examples
non_empty_examples = [ex for ex in validation_data if ex["labels"]]
empty_examples = [ex for ex in validation_data if not ex["labels"]]
few_shot_examples = random.sample(non_empty_examples, 4) + random.sample(empty_examples, 1)

In [43]:
manual_detailed_description = f"""
You are an assistant that extracts social group mentions from text.
                        
Definition of a social group:
A social group is a segment of society or a collection of people who share common socio-demographic traits or attributes that are ascriptive and/or acquired.
Institutionally organized groups and state authorities are not regarded as social groups. Groupings of individuals within these institutionally organized groups are however included as long as the defining feature of the group is a common socio-demographic trait or attribute. Groupings based on shared beliefs, life experiences, ideology, party affiliation and/or political opinion are excluded as well as references to the people within this group are excluded.
Mentions of general groupings like "everyone" "people" or "communities" are only regarded as a social group if the term appears with a modifier that indicates a specific and defined grouping.

Instructions on potential ambiguities:
- References that describe a group in abstract terms by highlighting a shared sociodemographic characteristic should be coded as a social group.
- Indirect group mentions as part of policy program, title, statute, or piece of legislation should be coded as social group.
- Social groups as part of composite terms should be coded as social group.
- Constituents or residents are not treated as implicit mentions but as social groups.
- Social group mentions generally occur in the plural form. If the text addresses one specific individual or very small specific collective such as one single family this is not a social group mention.
- Singular forms are included if it is a generalization about a broader group of people ("every", "any"). Only mark the group's name and not the modifying term.
- When the singular form of a group reference appears as part of a compound or composite word, the group's name should also be coded as a social group mention.

Instructions on the length of the mentions:
- When a group is mentioned using a genitive form, code the word without the genitive "'s".
- Definite and indefinite articles are generally not considered part of the group mention and should be excluded when coding.
- When multiple groups are mentioned and separated by a conjunction, and the reference to each group is clear without relying on the context of the others, each group should be coded separately.
- When multiple groups are mentioned together in a way that cannot be separated without losing the meaning of each individual reference, all mentions should be coded as a single social group.
- Numerical descriptors preceding a group reference are not considered part of the social group mention itself.
- When a text provides additional information about a group, this description should be included as part of the social group mention if it specifies the sociodemographic profile of the group.

Output format:
Return the full sentence. Mark the start of each social group mention with @@ and the end with ##. If there are no social group mentions, just respond with the full sentence without changing anything.
"""

manual_distilled_description

3079

In [44]:
# create different prompt templates for testing performance on the validation set
def compile_prompt(system_prompt, few_shot_examples, test_sentence):
        chat = [
                {
                        "role": "system",
                        "content": system_prompt
                }]

        for example in few_shot_examples:
                chat.append({"role": "user", "content": f"Sentence: {example['text']}"})
                chat.append({"role": "assistant", "content": example["llm_text"]})

        # add the test sentence
        chat.append({"role": "user", "content": f"Sentence: {test_sentence}"})

        return chat

In [45]:
# test an example sentence
test_sentence = "Our party supports the rights of young people and mothers!"
prompt = compile_prompt(manual_detailed_description, few_shot_examples, test_sentence)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=prompt
)

output_text = response.choices[0].message.content.strip()
output_text

'Our party supports the rights of @@young people## and @@mothers##!'

In [47]:
validation_data = validation_data[0:20]

# generate the answers for the normal format and store in a list
gen_answers = []

for i in range(len(validation_data)):
    sentence = validation_data[i]["text"]
    prompt = compile_prompt(manual_detailed_description, few_shot_examples, sentence)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=prompt
        )
    output_text = response.choices[0].message.content.strip()
    
    # convert the generated prediction to the bio scheme
    output_bio = llm_output_to_bio(output_text)
    gen_answers.append({"text": output_text,
                        "bio": output_bio})

In [55]:
for i in range(20):
    print(f"Sentence {i+1}")
    print(validation_data[i]["llm_text"])
    print(gen_answers[i]["text"])
    print("-"*100)

Sentence 1
Its economic plan highlighted the importance of signage in boosting business and tourism on South Hayling Island.
Its economic plan highlighted the importance of signage in boosting business and tourism on South Hayling Island.
----------------------------------------------------------------------------------------------------
Sentence 2
Will not this arrangement protect the incomes of @@lower paid barristers##?
Will not this arrangement protect the incomes of @@lower paid barristers##?
----------------------------------------------------------------------------------------------------
Sentence 3
We have dealt with-and continue to deal with-abuse in the @@student## visa system, which was allowed to increase significantly under the previous Labour Government, and non-EU migration is now at the levels of the late 1990s.
We have dealt with-and continue to deal with-abuse in the @@student## visa system, which was allowed to increase significantly under the previous Labour Govern

In [49]:
# evaluate the generated answers

# get list of bio tags only
ground_truth_bio = [[tag for (_, tag) in sent["bio_tags"]] for sent in validation_data]
pred_bio = [[tag for (_, tag) in sent["bio"]] for sent in gen_answers]

filtered_gt = []
filtered_pred = []
for gt, pred in zip(ground_truth_bio, pred_bio):
    if len(gt) == len(pred):
        filtered_gt.append(gt)
        filtered_pred.append(pred)

y_true = [tag for sent in filtered_gt for tag in sent]
y_pred = [tag for sent in filtered_pred for tag in sent]

# evaluate at the entity level with seqeval
print(seqeval_classification_report(filtered_gt, filtered_pred))

              precision    recall  f1-score   support

          sg       0.39      0.79      0.52        14

   micro avg       0.39      0.79      0.52        14
   macro avg       0.39      0.79      0.52        14
weighted avg       0.39      0.79      0.52        14



In [56]:
all_true_spans = []
all_predicted_spans = []

for idx in range(len(filtered_gt)):

    # get the spans
    all_true_spans.append(extract_spans(filtered_gt[idx]))
    all_predicted_spans.append(extract_spans(filtered_pred[idx]))

# apply cross-span evaluation
mention_level_evaluation(all_true_spans, all_predicted_spans)

{'precision': 0.43571428571428567, 'recall': 0.5, 'f1': 0.4499458874458874}